# 04. train_pool / val / test 나누기 + k-shot 학습 파일 만들기

**입력**: 03에서 만든 `processed/windows_A.jsonl`, `windows_B.jsonl`, 그리고 `labels.csv`

**출력** (드라이브 `splits/A/`, `splits/B/`)
- `pool.jsonl`, `val.jsonl`, `test.jsonl`: 나눈 결과 (우리 형식, label_id 포함)
- `kshot/train_k{k}_s{seed}.json`: 학습용 파일 (GLiClass `train.py` 형식) k = 1, 3, 5, 8, 16 × seed 0, 1, 2
- `val_gliclass.json`, `test_gliclass.json`: 평가용 파일 (GLiClass 형식)

**나누는 규칙 (앞에서 정한 것)**
1. 숨긴 전술(Defense Impairment, Impact, Collection)이 붙은 윈도우는 **전부 test**. 학습·검증에 절대 안 들어감
2. 윈도우가 20개 이상인 큰 세션은 **세션 안에서 시간 순서대로** 앞 50% pool, 다음 25% val, 마지막 25% test
3. 윈도우가 20개 미만인 작은 세션은 **세션 통째로** 한 곳에 넣음 (라벨별로 50/25/25에 가깝게)
4. 같은 세션에서 다른 칸으로 간 윈도우끼리 이벤트가 겹치면(윈도우 번호 차이 5 미만) **아래 칸 쪽을 버림** (test > val > pool 순으로 남김). 경계에서 4개씩 버리는 것과 같은 효과

**왜 전부 세션 단위로 못 나누나**: 정상(BENIGN) 윈도우가 세션 2개(aws_create_acl, aws_kms_key)에서만 나오고, Discovery도 큰 세션 3개가 대부분이라 세션 단위로만 나누면 어떤 칸에는 정상이 아예 없게 된다.

**준비**: 03을 돌려서 드라이브에 `processed/` 폴더가 있어야 함. CPU로 1분 이내.

## 0. 파일 찾기

In [ ]:
import os, glob, json
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    ROOT = '/content/drive/MyDrive'
except ImportError:
    ROOT = os.environ.get('DATA_DIR', '.')

lab_hits = glob.glob(f'{ROOT}/**/labels.csv', recursive=True)
win_hits = glob.glob(f'{ROOT}/**/processed/windows_A.jsonl', recursive=True)
assert lab_hits and win_hits, 'labels.csv 또는 processed/windows_A.jsonl을 못 찾음 (03 먼저)'
DATA_DIR = os.path.dirname(lab_hits[0])
PROC = os.path.dirname(win_hits[0])
print('DATA_DIR =', DATA_DIR, '| PROC =', PROC)

## 1. 불러오기

[볼 것] A 4,999개, B 4,236개면 03 결과 그대로.

In [ ]:
import pandas as pd, numpy as np
from collections import Counter
pd.set_option('display.width', 200)

labels = pd.read_csv(lab_hits[0])
TRAIN = labels[labels.role == 'train'].label_id.tolist()
HELD  = labels[labels.role == 'held_out'].label_id.tolist()
name  = dict(zip(labels.label_id, labels.name))
text_of = dict(zip(labels.label_id, labels.label_text))

data = {k: pd.read_json(f'{PROC}/windows_{k}.jsonl', lines=True) for k in 'AB'}
for k, d in data.items():
    d['key'] = d['labels'].apply('|'.join)
print({k: len(d) for k, d in data.items()})
print('학습 라벨:', TRAIN)
print('숨긴 라벨:', HELD)

## 2. 세션 구조 보기

[하는 일] 라벨마다 세션이 몇 개이고, 가장 큰 세션이 얼마나 차지하는지 본다.

[볼 것] BENIGN은 세션이 2개뿐. 그래서 큰 세션은 세션 안에서 시간으로 나눠야 한다.

[확인 질문] Initial Access and Stealth는 항상 Persistence and Privilege Escalation과 같이 붙어 나온다. 이게 학습에 어떤 문제가 될까? (힌트: 모델이 둘을 구분할 단서가 있을까)

In [ ]:
for k in 'AB':
    d = data[k]
    rows = []
    for lid in TRAIN + HELD:
        h = d[d['labels'].apply(lambda ls: lid in ls)]
        top = h.session.value_counts()
        rows.append(dict(label=lid, name=name[lid], role='train' if lid in TRAIN else 'held_out',
                         windows=len(h), sessions=len(top), biggest=top.iloc[0] if len(top) else 0))
    print(f'===== {k}'); print(pd.DataFrame(rows).to_string(index=False)); print()

print('라벨 조합별 개수 (B):'); print(data['B'].key.value_counts().to_string())

## 3. 나누기 (규칙 1~3)

[하는 일] 숨긴 라벨 → test, 큰 세션 → 시간 순서 50/25/25, 작은 세션 → 통째로 라벨별 비율 맞추기.

작은 세션 배정 방법: 크기가 큰 세션부터, 그 라벨 조합에서 목표 비율 대비 **가장 모자란 칸**에 넣는다.

In [ ]:
RATIO = {'pool': .50, 'val': .25, 'test': .25}
BIG = 20

def split(d):
    d = d.copy(); d['split'] = None
    held = d['labels'].apply(lambda ls: any(l in HELD for l in ls))
    d.loc[held, 'split'] = 'test'                                   # 규칙 1

    rest = d[~held]
    size = rest.groupby('session').size()
    for s in size[size >= BIG].index:                                # 규칙 2
        idx = rest[rest.session == s].sort_values('window').index
        c1, c2 = int(len(idx) * .50), int(len(idx) * .75)
        d.loc[idx[:c1], 'split'] = 'pool'; d.loc[idx[c1:c2], 'split'] = 'val'; d.loc[idx[c2:], 'split'] = 'test'

    small = size[size < BIG].sort_values(ascending=False).index      # 규칙 3
    main_key = rest.groupby('session').key.agg(lambda s: s.value_counts().index[0])
    total = rest.key.value_counts()
    for s in small:
        key = main_key[s]
        now = d[(d.key == key) & d.split.notna()].split.value_counts()
        need = {sp: RATIO[sp] * total[key] - now.get(sp, 0) for sp in RATIO}
        d.loc[(d.session == s) & d.split.isna(), 'split'] = max(need, key=need.get)
    assert d.split.notna().all()
    return d

parts = {k: split(data[k]) for k in 'AB'}
for k in 'AB':
    print(k, parts[k].split.value_counts().to_dict())

## 4. 겹치는 윈도우 버리기 (규칙 4)

[하는 일] 윈도우 크기가 5라서 번호 차이가 5 미만인 윈도우끼리는 같은 이벤트를 나눠 갖는다. 같은 세션에서 서로 다른 칸에 들어간 윈도우가 겹치면, 시험 쪽을 지키기 위해 아래 칸(pool 쪽)을 버린다.

[볼 것] 버리는 개수가 수십 개 수준이면 정상 (큰 세션 경계마다 4개 + 숨긴 라벨 주변).

[확인 질문] 이걸 안 하면 어떤 일이 생길까? (힌트: 학습 때 본 이벤트가 시험에도 나옴)

In [ ]:
RANK = {'pool': 0, 'val': 1, 'test': 2}

def purge(d):
    drop = []
    for s, g in d.groupby('session'):
        w = g.window.values; r = g.split.map(RANK).values
        for i, ix in enumerate(g.index):
            if ((np.abs(w - w[i]) < 5) & (r > r[i])).any():
                drop.append(ix)
    print(f'  겹쳐서 버림: {len(drop)}개', dict(d.loc[drop, 'split'].value_counts()))
    return d.drop(drop).reset_index(drop=True)

for k in 'AB':
    print(k); parts[k] = purge(parts[k])

## 5. 결과 표 + 검사

[볼 것]
- 학습 라벨은 pool에 16개 이상 있어야 k=16 가능
- 숨긴 라벨은 pool, val에 0개
- val, test에 학습 라벨이 골고루 있는지

[확인 질문] B의 Credential Access는 몇 개까지 k를 할 수 있나?

In [ ]:
for k in 'AB':
    d = parts[k]
    t = d.explode('labels').groupby(['labels', 'split']).size().unstack(fill_value=0).reindex(columns=['pool', 'val', 'test'], fill_value=0)
    t = t.reindex(TRAIN + HELD).fillna(0).astype(int)
    t.insert(0, 'name', [name[l] for l in t.index])
    t['max_k'] = [max([kk for kk in [1, 3, 5, 8, 16] if kk <= p], default=0) if l in TRAIN else '-' for l, p in zip(t.index, t.pool)]
    print(f'===== {k}  (총 {len(d)})'); print(t.to_string()); print()

    # 검사
    for sp in ['pool', 'val']:
        assert not d[d.split == sp]['labels'].apply(lambda ls: any(l in HELD for l in ls)).any(), f'{sp}에 숨긴 라벨!'
    assert d.text.is_unique
    for s, g in d.groupby('session'):
        w = g.sort_values('window'); sp = w.split.values; ww = w.window.values
        for i in range(len(w)):
            assert not ((np.abs(ww - ww[i]) < 5) & (sp != sp[i])).any(), f'겹침 남음: {s}'
print('검사 통과: 숨긴 라벨 누수 없음, 문장 중복 없음, 칸 사이 이벤트 겹침 없음')

## 6. k-shot 학습 파일 뽑기

[하는 일] pool에서 라벨마다 k개씩 뽑는다. seed(0, 1, 2)마다 다르게 뽑아서, 나중에 결과를 평균 낸다.

뽑는 방법: seed로 pool을 섞고, **드문 라벨부터** 그 라벨이 붙은 윈도우를 k개 찰 때까지 담는다. 라벨이 2개 붙은 윈도우는 두 라벨 모두에 1개로 센다.

pool에 k개가 안 되는 라벨이 있으면 그 k는 만들지 않고 알려준다.

[확인 질문] 왜 seed를 여러 개 쓸까? (힌트: k=1이면 어떤 1개를 뽑았냐에 따라 결과가 크게 흔들림)

In [ ]:
KS, SEEDS = [1, 3, 5, 8, 16], [0, 1, 2]

def sample_kshot(pool, k, seed):
    order = pool.sample(frac=1, random_state=seed)
    has = {l: order['labels'].apply(lambda ls: l in ls) for l in TRAIN}
    chosen, cnt = [], Counter()
    for l in sorted(TRAIN, key=lambda l: has[l].sum()):
        for ix in order.index[has[l]]:
            if cnt[l] >= k: break
            if ix not in chosen:
                chosen.append(ix); cnt.update(order.at[ix, 'labels'])
    short = [l for l in TRAIN if cnt[l] < k]
    return pool.loc[chosen], cnt, short

def to_gliclass(d, lids):
    return [{'text': r.text, 'all_labels': [text_of[l] for l in lids],
             'true_labels': [text_of[l] for l in r.labels if l in lids]} for r in d.itertuples()]

for k in 'AB':
    pool = parts[k][parts[k].split == 'pool']
    out = f'{DATA_DIR}/splits/{k}/kshot'; os.makedirs(out, exist_ok=True)
    print(f'===== {k}')
    for kk in KS:
        for sd in SEEDS:
            s, cnt, short = sample_kshot(pool, kk, sd)
            if short:
                print(f'  k={kk} seed={sd}: 만들지 않음 (부족: {[name[l] for l in short]})'); break
            json.dump(to_gliclass(s, TRAIN), open(f'{out}/train_k{kk}_s{sd}.json', 'w'), ensure_ascii=False)
            if sd == 0:
                print(f'  k={kk}: 윈도우 {len(s)}개, 라벨별 {dict(cnt)}')

## 7. 나눈 결과 + 평가 파일 저장

- `val_gliclass.json`: 후보 라벨 = 학습 라벨 5개 (숨긴 라벨 없음, 튜닝용)
- `test_gliclass.json`: 후보 라벨 = 학습 5개 + 숨긴 3개 = 8개 (본 라벨 / 안 본 라벨 성능을 같이 잼)

`pool/val/test.jsonl`에는 label_id가 그대로 있어서, 재영이가 평가 코드에서 본 라벨/숨긴 라벨을 나눠 계산할 때 이걸 쓰면 된다.

In [ ]:
cols = ['text', 'labels', 'session', 'window', 'source', 'split']
for k in 'AB':
    out = f'{DATA_DIR}/splits/{k}'
    for sp in ['pool', 'val', 'test']:
        parts[k][parts[k].split == sp][cols].to_json(f'{out}/{sp}.jsonl', orient='records', lines=True, force_ascii=False)
    json.dump(to_gliclass(parts[k][parts[k].split == 'val'], TRAIN), open(f'{out}/val_gliclass.json', 'w'), ensure_ascii=False)
    json.dump(to_gliclass(parts[k][parts[k].split == 'test'], TRAIN + HELD), open(f'{out}/test_gliclass.json', 'w'), ensure_ascii=False)
    print(k, sorted(os.listdir(out)), '| kshot 파일', len(os.listdir(f'{out}/kshot')), '개')

ex = json.load(open(f'{DATA_DIR}/splits/B/kshot/train_k1_s0.json'))[0]
print('\n학습 파일 한 줄 예시:'); print(' text:', ex['text'][:150]); print(' true_labels:', [t[:40] for t in ex['true_labels']]); print(' all_labels 개수:', len(ex['all_labels']))